# Visual Servoing with Foundation Models: evaluation

This notebook reads one evaluation campaign (`scripts/run_campaign.py experiments/final.yaml`,
README section 5) from `data/eval/<campaign>/` and produces the results the assignment asks for in
section 2.5: success rate, remaining pose and feature error, convergence time, error and
velocity over time, and the camera and feature trajectories. It needs no running simulation and
runs top to bottom on its own (`jupyter nbconvert --to notebook --execute --inplace`); the
campaign folder is `final` unless the environment variable `VS_CAMPAIGN` names another one.

**Setup.** UR5e with a wrist camera in the lab simulation. Grounding DINO finds the target from a
text prompt, a tracker (SAM 2 or CSRT) follows it between detector calls, and an IBVS controller
drives the camera into the reference pose 0.30 m above the target. The ground truth of the
simulation is used here only, never in the controller.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data/eval') / os.environ.get('VS_CAMPAIGN', 'final')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def load(block):
    """summary.csv of one campaign block (experiments/final.yaml), or None."""
    folder = DATA / block
    if not (folder / 'summary.csv').exists():
        return None
    df = pd.read_csv(folder / 'summary.csv')
    df['folder'] = str(folder)
    return df


def series(folder, target, start):
    return pd.read_csv(Path(folder) / 'runs' / f'{target}_S{start}.csv')

## 1. Metrics

| Metric | Definition |
|---|---|
| Success | converged within 60 s and final camera pose within 5 mm and 2 deg of the reference |
| Rotation error | ignores what the target's symmetry hides: round objects (can, cup, ball) have no observable yaw, the box is compared modulo 180 deg |
| Convergence time | from the start command until the feature error stays below 0.01 for 1 s |
| Final feature error | norm of the normalised image feature error at the end |

The tolerances are the runner's defaults (`--pos-tol`, `--rot-tol`).

## 2. Baseline: SAM 2 against CSRT as tracker

Same detector (Grounding DINO, one call per second), same 4 targets x 8 start configurations;
only the tracker that carries the loop between detector calls differs.

In [ ]:
base = {name: load(label) for name, label in
        [('SAM 2', 'main_dino_sam2'), ('CSRT', 'main_dino_csrt')]}
rows = []
for name, df in base.items():
    if df is None:
        continue
    for target, g in df.groupby('target'):
        rows.append({'tracker': name, 'target': target, 'success': f"{g.success.sum()}/{len(g)}",
                     'median pos err [mm]': round(g.final_pos_err_m.median() * 1000, 2),
                     'median rot err [deg]': round(g.final_rot_err_deg.median(), 2),
                     'median conv time [s]': round(g.conv_time_s.median(), 1)})
    rows.append({'tracker': name, 'target': 'all', 'success': f"{df.success.sum()}/{len(df)}",
                 'median pos err [mm]': round(df.final_pos_err_m.median() * 1000, 2),
                 'median rot err [deg]': round(df.final_rot_err_deg.median(), 2),
                 'median conv time [s]': round(df.conv_time_s.median(), 1)})
pd.DataFrame(rows).set_index(['tracker', 'target'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (col, scale, label) in zip(axes, [('final_pos_err_m', 1000, 'final position error [mm]'),
                                          ('conv_time_s', 1, 'convergence time [s]')]):
    data, names = [], []
    for name, df in base.items():
        if df is not None:
            data.append(df[col].dropna() * scale)
            names.append(name)
    ax.boxplot(data, tick_labels=names)
    ax.set_ylabel(label)
    if col == 'final_pos_err_m':
        ax.axhline(5, color='tab:red', ls='--', lw=1, label='tolerance 5 mm')
        ax.set_yscale('log')
        ax.legend()
fig.suptitle('Baseline, 4 targets x 8 start configurations')
fig.tight_layout()
fig.savefig(FIG / 'baseline_errors.png')

**Reading.** SAM 2 gives a mask, so the features include the orientation of the target and the
area does not change when the target rotates. CSRT tracks an axis-aligned box: for the rotated
box its area varies with the angle, so the distance estimate is wrong and the yaw is never
servoed (blue box 0/8). On the ball, the box around a shaded sphere ends a few millimetres off,
just above the tolerance.

## 3. Which start configurations converge reliably?

In [ ]:
df = base['SAM 2']
if df is not None:
    piv = df.pivot_table(index='target', columns='start', values='success', aggfunc='max')
    fig, ax = plt.subplots(figsize=(7, 2.6))
    ax.imshow(piv.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(piv.shape[1]), [f'S{s}' for s in piv.columns])
    ax.set_yticks(range(piv.shape[0]), piv.index)
    ax.grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, 'ok' if piv.values[i, j] else 'fail', ha='center', va='center', fontsize=8)
    ax.set_title('Success per start configuration (DINO + SAM 2)')
    fig.tight_layout()
    fig.savefig(FIG / 'start_configs.png')

Start configurations (from the lab's `targets.yaml`): S1 translation, S2 30 deg rotation, S3
combined, S4 far (+20 cm), S5 tilted, S6 90 deg rotation, S7 170 deg rotation, S8 close (-5 cm).
For S7 the box ends 180 deg off in absolute terms, which the symmetry rule counts as reached:
its view is identical to the reference.

## 4. One run in detail: error, velocity and trajectories

In [ ]:
def plot_run(folder, target, start, title):
    s = series(folder, target, start)
    fig = plt.figure(figsize=(12, 7))
    ax = fig.add_subplot(2, 2, 1)
    for col in ['e_x', 'e_y', 'e_area', 'e_angle']:
        if s[col].notna().any():
            ax.plot(s.t, s[col], label=col)
    ax.set_title('image feature error'); ax.set_xlabel('t [s]'); ax.legend()
    ax = fig.add_subplot(2, 2, 2)
    for col in ['v_x', 'v_y', 'v_z', 'w_z']:
        ax.plot(s.t, s[col], label=col)
    ax.set_title('commanded camera velocity [m/s, rad/s]'); ax.set_xlabel('t [s]'); ax.legend()
    # Top view plus height instead of a 3D axis: easier to read, and Ubuntu's mpl_toolkits
    # shadows the 3D toolkit of the venv's matplotlib in the container
    ax = fig.add_subplot(2, 2, 3)
    sc = ax.scatter(s.cam_x * 1000, s.cam_y * 1000, c=s.cam_z * 1000, s=4, cmap='viridis')
    ax.scatter(s.cam_x.iloc[0] * 1000, s.cam_y.iloc[0] * 1000, color='tab:green', label='start', zorder=3)
    ax.scatter(s.cam_x.iloc[-1] * 1000, s.cam_y.iloc[-1] * 1000, color='tab:red', label='end', zorder=3)
    fig.colorbar(sc, ax=ax, label='camera height z [mm]')
    ax.set_aspect('equal'); ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    ax.set_title('camera trajectory, top view (world)'); ax.legend()
    ax = fig.add_subplot(2, 2, 4)
    ok = s.feat_valid == 1
    ax.plot(s.u[ok], s.v[ok])
    ax.scatter(s.u[ok].iloc[0], s.v[ok].iloc[0], color='tab:green', label='start')
    ax.scatter(s.u[ok].iloc[-1], s.v[ok].iloc[-1], color='tab:red', label='end')
    ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal')
    ax.set_title('feature trajectory in the image [px]'); ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    return fig


df = base['SAM 2']
if df is not None:
    # blue_box S3 shows yaw and translation together; any other run if it is missing (smoke test)
    folder = Path(df.folder.iloc[0])
    target, start = ('blue_box', 3) if (folder / 'runs' / 'blue_box_S3.csv').exists() else         (df.target.iloc[0], int(df.start.iloc[0]))
    fig = plot_run(folder, target, start, f'{target}, S{start}, DINO + SAM 2')
    fig.savefig(FIG / f'run_{target}_S{start}.png')

## 5. Robustness: partial occlusion and lighting

A grey plate 1 cm above the target covers a fraction of its footprint (`--occlusion`), either
from the start (the detector has to find the partly hidden target) or appearing 2 s into the run
(`--occlusion-after`, the tracker has to keep a target it already follows). Alternatively both
lights of the world are scaled (`--light`). The reference is always captured without
disturbance.

In [ ]:
rob = {name: load(label) for name, label in [
    ('baseline', 'main_dino_sam2'), ('occlusion 20 %', 'rob_occl20'), ('occlusion 40 %', 'rob_occl40'),
    ('occlusion 20 % after 2 s', 'rob_occl20_mid'), ('occlusion 40 % after 2 s', 'rob_occl40_mid'),
    ('light 30 %', 'rob_light30'), ('light 150 %', 'rob_light150')]}
rows = [{'condition': k, 'success': f"{d.success.sum()}/{len(d)}",
         'success rate': round(d.success.mean(), 2),
         'median pos err [mm]': round(d.final_pos_err_m.median() * 1000, 2)}
        for k, d in rob.items() if d is not None]
table = pd.DataFrame(rows).set_index('condition')
table

In [ ]:
if len(table):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(table.index, table['success rate'], color='tab:blue')
    ax.set_ylim(0, 1); ax.set_ylabel('success rate')
    ax.set_title('Robustness (DINO + SAM 2, 4 targets x 8 starts each)')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    fig.tight_layout()
    fig.savefig(FIG / 'robustness.png')

## 6. Inference time

The detector period and an artificial delay per call (`--detector-period`,
`--detector-delay`) set the latency explicitly, so the result does not depend on the GPU the
evaluation runs on. The tracker bridges the time between detector calls; without it the controller
only gets features once per call.

In [ ]:
lat = {name: load(label) for name, label in [
    ('period 0.2 s', 'lat_p02'), ('period 1 s (baseline)', 'main_dino_sam2'),
    ('period 5 s', 'lat_p5'), ('period 5 s + 1 s delay', 'lat_p5_d1'),
    ('no tracker, period 1 s', 'lat_notracker')]}
rows = [{'configuration': k, 'success': f"{d.success.sum()}/{len(d)}",
         'median conv time [s]': round(d.conv_time_s.median(), 1),
         'median pos err [mm]': round(d.final_pos_err_m.median() * 1000, 2)}
        for k, d in lat.items() if d is not None]
pd.DataFrame(rows).set_index('configuration') if rows else 'no inference-time runs yet'

## 6b. Controller gain

`gain_*` blocks (`run_eval --gain`): the IBVS gain λ of `v = −λ L⁺ e` at 0.3, 0.6 (baseline), 1.0,
1.5 and 2.5 on `red_can` (no yaw) and `blue_box` (with yaw), all start configurations, same
detector and tracker as the baseline. Overshoot comes from the ground truth: the largest excursion
past the reference pose per axis, against the sign of the initial error, relative to it
(`vs_evaluation.gain_study`). The peak camera speed shows from which λ on the speed limit
(`max_linear_speed`, 0.1 m/s) takes over from the gain.

In [ ]:
import sys
sys.path.insert(0, '../student_ws/src/vs_evaluation')
from vs_evaluation import gain_study

gain_runs = gain_study.collect(DATA) if DATA.exists() else None
if gain_runs is not None and len(gain_runs) and gain_runs.gain.nunique() > 1:
    gain_table = gain_study.summarise(gain_runs)
    gain_study.plot(gain_table, FIG / 'gain_study.png')
    display(gain_table.round(3))
    plt.imshow(plt.imread(FIG / 'gain_study.png')); plt.axis('off'); plt.show()
else:
    print('no gain-study runs yet')

## 7. Safe stop when the target disappears

`--lose-target-after 3` moves the target out of view three seconds into a run. The requirement is
a safe stop; the runner reports the time until the camera stays within 1 mm and the drift.

In [ ]:
lost = load('lost')
lost[['target', 'start', 'lost_stop_time_s', 'drift_after_loss_m', 'success']] if lost is not None \
    else 'no safe-stop runs yet'

## 8. Localisation models in the closed loop

Every model seeds the same SAM 2 tracker from a box (or mask) and runs again only when the track
is lost, so the comparison isolates the localisation step. VLMs return no confidence; their box
is taken as is. Opus 5.5 runs through the Claude Code CLI (`scripts/claude_vlm_server.py`).

In [ ]:
DETS = {'Grounding DINO': 'main_dino_sam2', 'OWLv2': 'det_owlv2', 'YOLO-World': 'det_yolo_world',
        'Florence-2': 'det_florence2', 'SAM 3': 'det_sam3', 'LocateAnything-3B': 'det_locate_anything',
        'Qwen3-VL 8B': 'det_qwen3vl_8b', 'Gemma 4 E4B': 'det_gemma4_e4b', 
        'Qwen3-VL 4B': 'det_qwen3vl_4b', 'Gemma 4 E2B': 'det_gemma4_e2b', 'Opus 5.5': 'det_opus55'}
det = {k: load(b) for k, b in DETS.items()}
rows = [{'model': k, 'success': f"{d.success.sum()}/{len(d)}", 'success rate': round(d.success.mean(), 2),
         'median conv time [s]': round(d.conv_time_s.median(), 1),
         'median pos err [mm]': round(d.final_pos_err_m.median() * 1000, 2),
         'median detector calls': d.detector_calls.median(),
         'not detected': int(d.outcome.isin(['not detected', 'no reference']).sum())}
        for k, d in det.items() if d is not None]
det_table = pd.DataFrame(rows).set_index('model') if rows else None
det_table

In [ ]:
if det_table is not None:
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.bar(det_table.index, det_table['success rate'], color='tab:blue')
    ax.set_ylim(0, 1); ax.set_ylabel('success rate')
    ax.set_title('Closed loop per localisation model (+ SAM 2 tracker, 4 targets x 8 starts)')
    plt.setp(ax.get_xticklabels(), rotation=25, ha='right')
    fig.tight_layout()
    fig.savefig(FIG / 'detectors_closed_loop.png')

## 9. Offline benchmark: prompt formulations, localisation quality and inference time

Saved camera frames from the nominal, occlusion and lighting blocks, every visible object as a
test case. The ground-truth box is the projection of the object geometry with the simulated
camera pose; a localisation counts as correct at IoU >= 0.5 of the highest-scoring box.
Prompt levels: category (`can`), attribute (`red can`, the prompt of the closed-loop runs),
synonym (`red soda tin`), indirect description (`the tallest object on the table`), and negative
prompts for objects that are not in the scene (any box is a false positive). The latency is the
model alone on an RTX 3080 Ti laptop GPU (Opus: CLI round trip including network).

In [ ]:
bench_path = DATA / 'bench' / 'results.csv'
bench = pd.read_csv(bench_path) if bench_path.exists() else None
if bench is not None:
    pos = bench[bench.level != 'negative']
    acc = pos.pivot_table(index='detector', columns='level', values='correct', aggfunc='mean')
    acc = acc[[c for c in ['attribute', 'category', 'synonym', 'description'] if c in acc]]
    acc['false positives (negative prompts)'] = bench[bench.level == 'negative'].groupby('detector').false_positive.mean()
    acc['median IoU (attribute)'] = pos[pos.level == 'attribute'].groupby('detector').iou.median()
    acc['median latency [s]'] = bench.groupby('detector').latency_s.median()
    display(acc.round(2))
else:
    print('no offline benchmark yet')

In [ ]:
if bench is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    acc[[c for c in ['attribute', 'category', 'synonym', 'description'] if c in acc]].plot.bar(ax=axes[0])
    axes[0].set_ylim(0, 1); axes[0].set_ylabel('correct (IoU >= 0.5)'); axes[0].set_title('Prompt formulation')
    axes[0].legend(title='prompt level', ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.08), fontsize=8)
    plt.setp(axes[0].get_xticklabels(), rotation=25, ha='right')
    lat = bench.groupby('detector').latency_s.median().sort_values()
    axes[1].barh(lat.index, lat.values, color='tab:gray'); axes[1].set_xscale('log')
    axes[1].set_xlabel('median latency per call [s]'); axes[1].set_title('Inference time')
    fig.tight_layout()
    fig.savefig(FIG / 'bench_prompts_latency.png')
    cond = pos[pos.level == 'attribute'].pivot_table(index='detector', columns='condition',
                                                     values='correct', aggfunc='mean')
    display(cond.round(2))

## 10. Failure cases

In [ ]:
series_all = {**base, **{k: v for k, v in rob.items() if k != 'baseline'}}
frames = [d.assign(series=k) for k, d in series_all.items() if d is not None]
fails = pd.concat(frames)
fails = fails[fails.success == 0]
fails[['series', 'target', 'start', 'outcome', 'final_pos_err_m', 'final_rot_err_deg']]

## 11. Discussion

Points to cover in the presentation (assignment section 2.5):

- which start configurations converge reliably, and why S8 (close and tilted) is the hardest
- partial occlusion: the tracker follows the visible part, the controller centres the wrong point
- lighting: the detector and tracker are hardly affected in this simulation
- inference time: how much the tracker compensates for slow foundation-model calls
- localisation models: closed-loop success against offline accuracy and latency; which prompt
  formulations each model understands, and false positives on prompts for absent objects
- failure modes: target lost (the controller stops on purpose), false positives, the symmetry of
  round and rectangular objects, the stalled simulation bridge under WSL
- limits of the approach: features from a mask or a box, no depth sensor, a fixed reference view